# Step 4: Correlate Distance and Genelists

This step computes distance/neighbor correlations to different gene lists

## Setup and imports

In [ ]:
# Imports
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import squidpy as sq
import pandas as pd
import numpy as np
import anndata as ad
import scipy.stats
from scipy.stats import wilcoxon
from matplotlib.patches import Patch
from statsmodels.stats.multitest import fdrcorrection

In [ ]:
# Reproducibility settings
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Path config
indir = '/path/to/integrated/processed_data/'
outdir = '/path/to/integrated/out/'
figdir = '/path/to/integrated/figures/'
genelist_file = '/path/to/genelists_inpanel.csv' # extended_data_fig3.xlsx '3d-i marker lists'

In [ ]:
# Figure settings
plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams["font.family"] = "Helvetica"
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

In [ ]:
# Color palettes
# Patients
patient_colors = {
    'Patient1': "#d9d9d9",
    'Patient4': "#bdbdbd",
    'Patient5': "#969696",
    'Patient2': "#636363",
    'Patient3': "#252525"
}

# T cell subtype
palette_map = {
    "Cytotoxic CD8T": "#ffffcc",
    "Naive/CM T": "#a1dab4",
    "Proliferating CD8T": "#41b6c4",
    "Treg": "#2c7fb8",
    "γδT": "#253494",
}

# Helper functions

In [ ]:
def get_neighbor_composition(adata, cell_mask, label_col="celltype"):
    # Sparse adjacency matrix of spatial neighbors (one row per cell)
    connectivities = adata.obsp["spatial_connectivities"]
    target_indices = np.where(cell_mask)[0]

    if len(target_indices) == 0:
        return None

    all_neighbors = []
    labels = adata.obs[label_col].values

    # For each focal cell, collect labels of its graph neighbors.
    for idx in target_indices:
        neighbor_indices = connectivities[idx].indices
        all_neighbors.extend(labels[neighbor_indices])

    if not all_neighbors:
        return None

    # Return normalized proportions of neighbors
    counts = pd.Series(all_neighbors).value_counts(normalize=True)
    return counts

In [ ]:
def make_df_plot(sample_results, target_subtypes, top_n=10):
    """
    sample_results: dict[subtype] = pd.Series(neighbor_type -> proportion)
    returns df_plot:
      index   = subtype (x-axis)
      columns = neighbor cell types (stacked bars)
      values  = proportions
    """
    if not sample_results:
        return None

    df_res = pd.DataFrame(sample_results).fillna(0)  # rows=neighbor types, cols=subtypes

    top_types = df_res.mean(axis=1).nlargest(top_n).index.tolist()
    df_top = df_res.loc[top_types].copy()

    other_row = 1.0 - df_top.sum(axis=0)
    other_row = other_row.clip(lower=0)  # avoid tiny negative due to float roundoff
    df_top.loc["Other"] = other_row

    df_plot = df_top.T  # rows=subtypes, cols=neighbor types
    df_plot = df_plot.reindex(target_subtypes).fillna(0)

    return df_plot

In [ ]:
def plot_neighbor_composition(
    df_plot,
    title,
    outpath,
    color_dict,
    x_categories,
    show_legend=True,
    figsize=(5/2.54, 8/2.54),
    
):
    # Force every plot to have identical x-axis categories/positions
    df_plot = df_plot.reindex(x_categories).fillna(0)

    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(x_categories))

    for i, subtype in enumerate(x_categories):
        row = df_plot.loc[subtype]

        ordered = list(df_plot.columns)
        #ordered = row.sort_values(ascending=False).index.tolist()
        #if "Other" in ordered:
        #    ordered.remove("Other")
        #    ordered.append("Other")

        bottom = 0.0
        for col in ordered:
            val = row[col]
            if val <= 0:
                continue

            ax.bar(
                x[i], val, bottom=bottom,
                color=color_dict.get(col, "#E6E6E6"),
                edgecolor="white", linewidth=0.5, width=0.95
            )

            if val >= 0.05:
                ax.text(
                    x[i], bottom + val / 2, f"{val:.2f}",
                    ha="center", va="center"
                )
            bottom += val

    ax.set_title(title, pad=20)
    ax.set_xlabel("T Cell Subtype")
    ax.set_ylabel("Proportion of Neighbors")

    # Fixed ticks for all plots
    ax.set_xticks(x)
    ax.set_xticklabels(x_categories, rotation=90, ha="right")
    ax.set_xlim(-0.5, len(x_categories) - 0.5)

    ax.set_ylim(0, 1.0)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.set_yticklabels([f"{v:.1f}" for v in np.linspace(0, 1, 6)])
    ax.tick_params(axis="both", which="both", left=True, width=0.5, length=2)

    ax.spines["bottom"].set_linewidth(0.5)
    ax.spines["bottom"].set_visible(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    if show_legend:
        legend_handles = [
            Patch(facecolor=color_dict.get(col, "#E6E6E6"), edgecolor="white", label=col)
            for col in df_plot.columns
        ]
        ax.legend(
            handles=legend_handles, title="Neighbor Cell Type",
            bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False
        )

    plt.tight_layout()
    plt.savefig(outpath, format="pdf", transparent=True)
    plt.show()

## Load data

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')

In [ ]:
genelist_df = pd.read_csv(genelist_file)

## T-cell-related gene lists vs timepoints

In [ ]:
# Choose columns 0-2 based on gene list of interest
tcell_genes = genelist_df.iloc[:, 2].dropna().tolist() # 0, 1 or 2
tcell_genes = [str(g).strip() for g in tcell_genes[1:] if str(g).strip()]

# Extract gene list name from row (header)
gene_list_name = genelist_df.columns[2] #0, 1 or 2
score_name = gene_list_name.replace(' ', '_')

# Score T cells only
t_cells = adata[adata.obs['celltype'] == 'T'].copy()

# Score the genes and adds to t_cells.obs
sc.tl.score_genes(t_cells, gene_list=tcell_genes, score_name=score_name)

In [ ]:
# Calculate gene score medians
medians = (
    t_cells.obs
    .groupby('T_subtype')[score_name]
    .median()
    .sort_values(ascending=False)
)
order = medians.index.tolist()

In [ ]:
# Save source data for boxplot
boxplot_df = (
    t_cells.obs[[ "T_subtype", score_name ]]
    .rename(columns={score_name: "score"})
    .dropna(subset=["T_subtype", "score"])
    .copy()
)

boxplot_df.to_csv(f"{outdir}boxplot_source_data_til_dysfunction.csv", index=False)

In [ ]:
# Make boxplot
fig, ax = plt.subplots(figsize=(4/2.54, 5/2.54))
sns.boxplot(
    data=t_cells.obs,
    x='T_subtype',
    y=score_name,
    palette=palette_map,
    showfliers=False,
    order=order,
    ax=ax,
    linewidth=0.5,
    boxprops={"linewidth": 0.5},
    whiskerprops={"linewidth": 0.5},
    capprops={"linewidth": 0.5},
    medianprops={"linewidth": 0.5}
)

ax.set_ylabel("Score")

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right')
ax.tick_params(axis="both", which="both", left=True, width=0.5, length=2)

ax.set_frame_on(False)
ax.grid(False)

plt.tight_layout()

plt.savefig(f'{figdir}boxplot_{score_name}_across_T_subtypes.pdf', 
            format='pdf', 
            transparent=True, 
            bbox_inches='tight')

plt.show()

In [ ]:
# Calculate mean score per sample (pseudobulk)
data = []
samples = t_cells.obs['sample'].unique()

for s in samples:
    sub = t_cells[(t_cells.obs['sample'] == s) & (t_cells.obs['celltype'] == 'T')]
    
    if sub.n_obs < 10: continue # Skip if too few T cells
    
    mean_score = sub.obs[score_name].mean()
    
    patient = sub.obs['patient'].iloc[0]
    timepoint = sub.obs['timepoint'].iloc[0]
    
    data.append({
        'Sample': s,
        'Patient': patient,
        'Timepoint': timepoint,
        'Score': mean_score
    })

df_score = pd.DataFrame(data)

In [ ]:
# Statistics (paired, matched by patient)
# Build paired table
paired = (
    df_score.pivot_table(index="Patient", columns="Timepoint", values="Score", aggfunc="mean")
       .dropna(subset=["DX", "PT"])
)

# Within-patient difference (PT - DX)
diff = paired["PT"] - paired["DX"]
delta = diff.mean()  # mean paired change

# Paired t-test
from scipy.stats import ttest_rel, wilcoxon

t_res = ttest_rel(paired["PT"], paired["DX"], nan_policy="omit")
p_val = t_res.pvalue

# Robustness check for tiny n
try:
    w_res = wilcoxon(paired["PT"], paired["DX"], alternative="two-sided")
    p_val_wilcoxon = w_res.pvalue
except ValueError:
    p_val_wilcoxon = np.nan

sig_text = f"paired t p = {p_val:.2e}" if pd.notna(p_val) else "paired t p = N/A"

print(f"n pairs = {len(paired)}")
print(f"mean PT-DX (delta) = {delta:.4f}")
print(f"paired t-test p = {p_val:.4g}")
print(f"wilcoxon p = {p_val_wilcoxon:.4g}" if pd.notna(p_val_wilcoxon) else "wilcoxon p = N/A")

In [ ]:
# Build plotting dataframe from paired table
plot_df = (
    paired.reset_index()[["Patient", "DX", "PT"]]
          .melt(id_vars="Patient", value_vars=["DX", "PT"],
                var_name="Timepoint", value_name="Score")
)

plot_df["Timepoint"] = pd.Categorical(
    plot_df["Timepoint"], categories=["DX", "PT"], ordered=True
)
plot_df = plot_df.sort_values(["Patient", "Timepoint"])

In [ ]:
fig, ax = plt.subplots(figsize=(4/2.54, 5/2.54))

# Plot individual patient trajectories
for patient in plot_df["Patient"].unique():
    patient_data = plot_df[plot_df["Patient"] == patient]
    ax.plot(
        patient_data["Timepoint"], patient_data["Score"],
        "o-", alpha=1,
        color=patient_colors.get(patient, "gray"),
        linewidth=1.5,
        markersize=4,
        label="_nolegend_"
    )

ax.set_ylabel("Mean score")
ax.tick_params(axis="both", which="both", left=True, width=0.5, length=2)
ax.grid(False)
ax.set_frame_on(False)

# Legend
ax.legend(
    loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False
)

text = (
    f"n={len(paired)}\n"
    f"paired t p={p_val:.2e}\n"
    f"wilcoxon p={p_val_wilcoxon:.2e}" if pd.notna(p_val_wilcoxon)
    else f"n={len(paired)}\npaired t p={p_val:.2e}\nwilcoxon p=N/A"
)
if pd.notna(delta):
    text += f"\nΔ={delta:.2f}"

ax.text(
    0.5, 1.25, text, transform=ax.transAxes,
    verticalalignment="top",
    horizontalalignment="center"
)

plt.tight_layout()
plt.savefig(
    f"{figdir}pointplot_{score_name}_across_timepoints.pdf",
    format="pdf", transparent=True, dpi=600, bbox_inches="tight"
)
plt.show()

# Print data
print(paired)

In [ ]:
# Save paired table
paired.to_csv(f"{outdir}fig2f_table.csv", index=True) # 2f, 2h, 2j

In [ ]:
summary_df = pd.DataFrame({
    "metric": [
        "n_pairs",
        "mean_PT_minus_DX_delta",
        "paired_ttest_p",
        "wilcoxon_p",
    ],
    "value": [
        len(paired),
        float(delta),
        float(p_val) if pd.notna(p_val) else np.nan,
        float(p_val_wilcoxon) if pd.notna(p_val_wilcoxon) else np.nan,
    ],
})

summary_df.to_csv(f"{outdir}fig2f_stats_summary.csv", index=False) # 2f, 2h, 2j
summary_df

In [ ]:
# FDR correction
files = [
    f"{outdir}fig2f_stats_summary.csv",
    f"{outdir}fig2h_stats_summary.csv",
    f"{outdir}fig2j_stats_summary.csv",
]

dfs = []
for fp in files:
    df = pd.read_csv(fp)
    df["source_file"] = fp.split("/")[-1]   # keep track of which gene list
    dfs.append(df)

all_stats = pd.concat(dfs, ignore_index=True)
all_stats

In [ ]:
# All_stats columns expected: source_file, metric, value
all_stats["value"] = pd.to_numeric(all_stats["value"], errors="coerce")

# Paired t-test p-values across the 3 gene-list files
t = all_stats.loc[all_stats["metric"] == "paired_ttest_p", ["source_file", "value"]].copy()
t = t.rename(columns={"value": "paired_ttest_p"})
_, t["paired_ttest_p_fdr"] = fdrcorrection(t["paired_ttest_p"].values, alpha=0.05, method="indep")

# Wilcoxon p-values across the 3 gene-list files
w = all_stats.loc[all_stats["metric"] == "wilcoxon_p", ["source_file", "value"]].copy()
w = w.rename(columns={"value": "wilcoxon_p"})
_, w["wilcoxon_p_fdr"] = fdrcorrection(w["wilcoxon_p"].values, alpha=0.05, method="indep")

# Combine
fdr_table = t.merge(w, on="source_file", how="outer")
display(fdr_table)

# Save
fdr_table.to_csv(f"{outdir}fig2fhj_fdr.csv", index=False)

## Compute top 10 neighbors

In [ ]:
# Setup
samples_to_plot = list(adata.obs["sample"].unique())

target_subtypes = ['γδT', 'Treg', 'Naive/CM T',  'Proliferating T', 'Cytotoxic T']

celltype_categories = list(adata.obs["celltype"].cat.categories)
celltype_colors = list(adata.uns["celltype_colors"])
color_dict = dict(zip(celltype_categories, celltype_colors))
color_dict["Other"] = "#E6E6E6"

weighting = "equal"  # or "cell_count"

In [ ]:
# Compute per-sample results
sample_to_results = {}
results_by_subtype = {st: [] for st in target_subtypes}
weights_by_subtype = {st: [] for st in target_subtypes}

for sample in samples_to_plot:
    print(f"\nProcessing {sample}...")
    adata_sub = adata[adata.obs["sample"] == sample].copy()
    if adata_sub.n_obs < 10:
        continue

    sq.gr.spatial_neighbors(adata_sub, coord_type="generic", spatial_key="spatial", n_neighs=10)

    sample_results = {}
    for subtype in target_subtypes:
        is_subtype = adata_sub.obs["T_subtype"] == subtype
        n_subtype = int(is_subtype.sum())
        if n_subtype < 10:
            continue

        res = get_neighbor_composition(adata_sub, is_subtype, label_col="celltype")
        if res is None:
            continue

        sample_results[subtype] = res
        results_by_subtype[subtype].append(res)
        weights_by_subtype[subtype].append(1.0 if weighting == "equal" else float(n_subtype))

    if sample_results:
        sample_to_results[sample] = sample_results

In [ ]:
# Plot individual sample figures
for sample, sample_results in sample_to_results.items():
    df_plot = make_df_plot(sample_results, target_subtypes, top_n=10)
    if df_plot is None:
        continue

    plot_neighbor_composition(
        df_plot=df_plot,
        title=f"Top10 Neighbor by T: {sample}",
        outpath=f"{figdir}top10neighbor_T_{sample}_celltype.pdf",
        color_dict=color_dict,
        x_categories=target_subtypes,
        show_legend=False,
    )

In [ ]:
# Build + plot consensus figure (if order by most to least frequent neighbor)
consensus_results = {}
for subtype, series_list in results_by_subtype.items():
    if not series_list:
        continue
    df = pd.concat(series_list, axis=1).fillna(0)
    w = np.array(weights_by_subtype[subtype], dtype=float)
    consensus_results[subtype] = (df * w).sum(axis=1) / w.sum()

if consensus_results:
    df_plot_cons = make_df_plot(consensus_results, target_subtypes, top_n=10)
    if df_plot_cons is not None:
        df_plot_cons = df_plot_cons.reindex(target_subtypes).fillna(0)

        plot_neighbor_composition(
            df_plot=df_plot_cons,
            title="Top10 Neighbor by T: Consensus",
            outpath=f"{figdir}top10neighbor_T_consensus_celltype.pdf",
            color_dict=color_dict,
            x_categories=target_subtypes,   # required for fixed x-tick positions
            show_legend=False,
        )
    else:
        print("Consensus plot dataframe is empty.")
else:
    print("No valid T subtypes found for consensus.")

In [ ]:
# Build + plot consensus figure (if order by celltype)
consensus_results = {}
for subtype, series_list in results_by_subtype.items():
    if not series_list:
        continue
    df = pd.concat(series_list, axis=1).fillna(0)
    w = np.array(weights_by_subtype[subtype], dtype=float)
    consensus_results[subtype] = (df * w).sum(axis=1) / w.sum()

if consensus_results:
    df_plot_cons = make_df_plot(consensus_results, target_subtypes, top_n=10)
    if df_plot_cons is not None:
        df_plot_cons = df_plot_cons.reindex(target_subtypes).fillna(0)

        # Neighbor segments: category order (matches adata celltype order / color_dict), Other last
        stack_cols = [c for c in celltype_categories if c in df_plot_cons.columns]
        if "Other" in df_plot_cons.columns:
            stack_cols = [c for c in stack_cols if c != "Other"] + ["Other"]
        # Any unexpected column (should be rare): keep after known order
        extra = [c for c in df_plot_cons.columns if c not in stack_cols]
        stack_cols = stack_cols + extra
        df_plot_cons = df_plot_cons[stack_cols]

        plot_neighbor_composition(
            df_plot=df_plot_cons,
            title="Top10 Neighbor by T: Consensus",
            outpath=f"{figdir}top10neighbor_T_consensus_celltype2.pdf",
            color_dict=color_dict,
            x_categories=target_subtypes,
            show_legend=False,
        )
    else:
        print("Consensus plot dataframe is empty.")
else:
    print("No valid T subtypes found for consensus.")

In [ ]:
# Build source-data rows for all individual sample plots + consensus plot
source_rows = []
top_n = 10

# Individual sample plots
for sample, sample_results in sample_to_results.items():
    df_plot = make_df_plot(sample_results, target_subtypes, top_n=top_n)
    if df_plot is None:
        continue

    # match plotting behavior
    df_plot = df_plot.reindex(target_subtypes).fillna(0)

    # long format (best for source data tables)
    long_df = (
        df_plot.reset_index(names="T_subtype")
        .melt(id_vars="T_subtype", var_name="Neighbor_CellType", value_name="Proportion")
    )
    long_df["Plot_Type"] = "individual"
    long_df["Plot_ID"] = sample
    long_df["Top_N"] = top_n
    long_df["Weighting"] = weighting
    source_rows.append(long_df)

# Consensus plot
consensus_results = {}
for subtype, series_list in results_by_subtype.items():
    if not series_list:
        continue
    df = pd.concat(series_list, axis=1).fillna(0)
    w = np.array(weights_by_subtype[subtype], dtype=float)
    consensus_results[subtype] = (df * w).sum(axis=1) / w.sum()

if consensus_results:
    df_plot_cons = make_df_plot(consensus_results, target_subtypes, top_n=top_n)
    if df_plot_cons is not None:
        df_plot_cons = df_plot_cons.reindex(target_subtypes).fillna(0)

        long_cons = (
            df_plot_cons.reset_index(names="T_subtype")
            .melt(id_vars="T_subtype", var_name="Neighbor_CellType", value_name="Proportion")
        )
        long_cons["Plot_Type"] = "consensus"
        long_cons["Plot_ID"] = "consensus"
        long_cons["Top_N"] = top_n
        long_cons["Weighting"] = weighting
        source_rows.append(long_cons)

# Combine and save
source_data = pd.concat(source_rows, ignore_index=True)
source_data = source_data[source_data["Neighbor_CellType"] != "Other"].copy()

# optional ordering
source_data = source_data[
    ["Plot_Type", "Plot_ID", "T_subtype", "Neighbor_CellType", "Proportion", "Top_N", "Weighting"]
]

source_data_wide = source_data.pivot_table(
    index=["Plot_Type", "Plot_ID", "T_subtype", "Top_N", "Weighting"],
    columns="Neighbor_CellType",
    values="Proportion",
    aggfunc="first",
    fill_value=0,
).reset_index()

source_data_wide.to_csv(f"{outdir}fig4_top10neighbor_11plots_source_data_wide.csv", index=False)

In [ ]:
# Save labeled adata
adata.write_h5ad(outdir + 'xenium_integrated_labeled.h5ad', compression='gzip')